# Loan Dataset Exploration with PySpark

This notebook is set up for the large `data/loan.csv` file using a local Spark session.

Prerequisites:
- Activate the virtual environment in the project root.
- Install the packages from `requirements.txt`.
- Install a Java runtime, since PySpark needs Java to start.


In [3]:
import os
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JAVA_HOME = Path('/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home')
os.environ.setdefault('JAVA_HOME', str(JAVA_HOME))
os.environ['PATH'] = f"{JAVA_HOME / 'bin'}:{os.environ['PATH']}"
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
CSV_PATH = DATA_DIR / 'loan.csv'
DICT_PATH = DATA_DIR / 'LCDataDictionary.xlsx'

spark = (
    SparkSession.builder
    .appName('loan-exploration')
    .master('local[*]')
    .config('spark.sql.repl.eagerEval.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'CSV exists: {CSV_PATH.exists()} | size: {CSV_PATH.stat().st_size / (1024 ** 3):.2f} GB')
print(f'Data dictionary exists: {DICT_PATH.exists()}')


Spark version: 4.1.1
CSV exists: True | size: 1.11 GB
Data dictionary exists: True


In [4]:
dictionary_preview = pd.read_excel(DICT_PATH).head(10)
dictionary_preview

,LoanStatNew,Description
0,acc_now_delinq,The number of accounts on which the borrower i...
1,acc_open_past_24mths,Number of trades opened in past 24 months.
2,addr_state,The state provided by the borrower in the loan...
3,all_util,Balance to credit limit on all trades
4,annual_inc,The self-reported annual income provided by th...
5,annual_inc_joint,The combined self-reported annual income provi...
6,application_type,Indicates whether the loan is an individual ap...
7,avg_cur_bal,Average current balance of all accounts
8,bc_open_to_buy,Total open to buy on revolving bankcards.
9,bc_util,Ratio of total current balance to high credit/...


In [22]:
df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(CSV_PATH))
)

df.cache()
row_count = df.count()
column_count = len(df.columns)
print(f'Rows: {row_count:,}')
print(f'Columns: {column_count}')


[Stage 45:=====>                                                   (1 + 9) / 10]

Rows: 2,260,668
Columns: 145


26/05/20 21:59:55 WARN CacheManager: Asked to cache already cached data.        


## Step 1: Initial Standardization

These cells keep `df` as the raw dataset and build `df_step1` as the first cleaned working frame.


In [23]:
string_columns = [name for name, dtype in df.dtypes if dtype == 'string']

df_trimmed = df.select([
    F.trim(F.col(c)).alias(c) if c in string_columns else F.col(c)
    for c in df.columns
])

df.filter(F.col('term') != F.trim(F.col('term'))).select('term').distinct().show(truncate=False)

+----------+
|term      |
+----------+
| 36 months|
| 60 months|
+----------+



In [24]:
placeholder_strings = ['', 'NULL', 'null', 'N/A', 'n/a']

df_nulls = df_trimmed.select([
    F.when(F.col(c).isin(placeholder_strings), None).otherwise(F.col(c)).alias(c)
    if c in string_columns else F.col(c)
    for c in df_trimmed.columns
])

df_nulls.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c)
    for c in ['id', 'member_id', 'url', 'desc']
]).show()

+-------+---------+-------+-------+
|     id|member_id|    url|   desc|
+-------+---------+-------+-------+
|2260668|  2260668|2260667|2134855|
+-------+---------+-------+-------+



In [25]:
term_digits = F.regexp_extract(F.col('term'), r'(\d+)', 1)

df_term = df_nulls.withColumn(
    'term',
    F.when(F.col('term').isNull(), None)
    .when(term_digits == '', None)
    .otherwise(term_digits.cast('int'))
)

df_term.groupBy('term').count().orderBy('term').show()

+----+-------+
|term|  count|
+----+-------+
|  36|1609754|
|  60| 650914|
+----+-------+



In [26]:
emp_length_digits = F.regexp_extract(F.col('emp_length'), r'(\d+)', 1)

df_emp_length = df_term.withColumn(
    'emp_length',
    F.when(F.col('emp_length').isNull(), None)
    .when(F.col('emp_length') == '< 1 year', F.lit(0))
    .when(F.col('emp_length') == '10+ years', F.lit(10))
    .when(emp_length_digits == '', None)
    .otherwise(emp_length_digits.cast('int'))
)

df_emp_length.groupBy('emp_length').count().orderBy('emp_length').show(15)

+----------+------+
|emp_length| count|
+----------+------+
|      NULL|146908|
|         0|189988|
|         1|148403|
|         2|203676|
|         3|180753|
|         4|136605|
|         5|139698|
|         6|102628|
|         7| 92695|
|         8| 91914|
|         9| 79395|
|        10|748005|
+----------+------+



In [27]:
zip_prefix = F.regexp_extract(F.col('zip_code'), r'^(\d{3})', 1)

df_step1 = df_emp_length.withColumn(
    'zip_code',
    F.when(F.col('zip_code').isNull(), None)
    .when(zip_prefix == '', None)
    .otherwise(zip_prefix)
)

df_step1.select('zip_code').show(15, truncate=False)

+--------+
|zip_code|
+--------+
|109     |
|713     |
|490     |
|985     |
|212     |
|461     |
|606     |
|460     |
|327     |
|068     |
|711     |
|300     |
|840     |
|278     |
|413     |
+--------+
only showing top 15 rows


## Step 2: Parse Month-Year Fields To Dates

These columns use the `MMM-YYYY` pattern and will be normalized to the first day of the month.


In [37]:
date_columns = [
    'issue_d',
    'earliest_cr_line',
    'last_pymnt_d',
    'next_pymnt_d',
    'last_credit_pull_d',
]

date_pattern = r'^[A-Za-z]{3}-\d{4}$'

df_step2 = df_step1

for c in date_columns:
    df_step2 = (
        df_step2
        .withColumn(f'{c}_invalid_raw', F.col(c).isNotNull() & (~F.col(c).rlike(date_pattern)))
        .withColumn(
            c,
            F.when(F.col(c).rlike(date_pattern), F.to_date(F.col(c), 'MMM-yyyy'))
             .otherwise(None)
        )
    )

In [35]:
df_step2.select(
    'issue_d', 'earliest_cr_line', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d'
).printSchema()

df_step2.select(
    'issue_d', 'earliest_cr_line', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d'
).show(10, truncate=False)

root
 |-- issue_d: date (nullable = true)
 |-- earliest_cr_line: date (nullable = true)
 |-- last_pymnt_d: date (nullable = true)
 |-- next_pymnt_d: date (nullable = true)
 |-- last_credit_pull_d: date (nullable = true)

+----------+----------------+------------+------------+------------------+
|issue_d   |earliest_cr_line|last_pymnt_d|next_pymnt_d|last_credit_pull_d|
+----------+----------------+------------+------------+------------------+
|2018-12-01|2001-04-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|1987-06-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|2011-04-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|2006-02-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|2000-12-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|2002-09-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|2004-11-01      |2019-02-01  |2019-03-01  |2019-02-01        |
|2018-12-01|1997-11-01      |

In [4]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: strin

In [107]:
df.select(df.columns[87:94]).show(15, truncate=False)

+---------------------+------------------------------+---------------------+--------------+---------------+-----------+---------+
|mths_since_recent_inq|mths_since_recent_revol_delinq|num_accts_ever_120_pd|num_actv_bc_tl|num_actv_rev_tl|num_bc_sats|num_bc_tl|
+---------------------+------------------------------+---------------------+--------------+---------------+-----------+---------+
|2                    |NULL                          |0                    |2             |5              |3          |3        |
|4                    |NULL                          |0                    |2             |4              |4          |9        |
|14                   |NULL                          |0                    |0             |3              |3          |3        |
|5                    |NULL                          |0                    |1             |2              |1          |2        |
|13                   |NULL                          |0                    |2             

In [108]:
missing_summary = (
    df.select([
        F.sum(F.col(c).isNull().cast('int')).alias(c)
        for c in df.columns
    ])
    .toPandas()
    .T
    .reset_index()
)

missing_summary.columns = ['column', 'missing_rows']
missing_summary['missing_pct'] = missing_summary['missing_rows'] / row_count
missing_summary.sort_values('missing_pct', ascending=False).head(20)

,column,missing_rows,missing_pct
0,id,2260668,1.000000
1,member_id,2260668,1.000000
18,url,2260667,1.000000
134,orig_projected_additional_accrued_interest,2252240,0.996272
136,hardship_last_payment_amount,2250054,0.995305
135,hardship_payoff_balance_amount,2250054,0.995305
133,hardship_loan_status,2250048,0.995302
132,hardship_dpd,2250048,0.995302
131,hardship_length,2250044,0.995301
128,hardship_start_date,2250044,0.995301


In [109]:
candidate_targets = [c for c in ['loan_status', 'grade', 'sub_grade', 'purpose', 'home_ownership'] if c in df.columns]
candidate_targets

['loan_status', 'grade', 'sub_grade', 'purpose', 'home_ownership']

In [110]:
if 'loan_status' in df.columns:
    (
        df.groupBy('loan_status')
        .count()
        .orderBy(F.desc('count'))
        .show(20, truncate=False)
    )
else:
    print('`loan_status` is not present in the dataset.')


+---------------------------------------------------+-------+
|loan_status                                        |count  |
+---------------------------------------------------+-------+
|Fully Paid                                         |1041952|
|Current                                            |919695 |
|Charged Off                                        |261654 |
|Late (31-120 days)                                 |21897  |
|In Grace Period                                    |8952   |
|Late (16-30 days)                                  |3737   |
|Does not meet the credit policy. Status:Fully Paid |1988   |
|Does not meet the credit policy. Status:Charged Off|761    |
|Default                                            |31     |
|Oct-2015                                           |1      |
+---------------------------------------------------+-------+



In [111]:
numeric_candidates = [
    c for c, t in df.dtypes
    if t in {'int', 'bigint', 'float', 'double', 'decimal'}
]

df.select(numeric_candidates[:15]).summary().show(truncate=False)

[Stage 117:>                                                        (0 + 1) / 1]

+-------+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------------+------------------+------------------+--------------------+---------------------+-----------------+------------------+--------------------+
|summary|loan_amnt         |funded_amnt       |funded_amnt_inv   |int_rate          |installment       |acc_open_past_24mths|bc_open_to_buy    |chargeoff_within_12_mths|delinq_amnt       |mo_sin_old_il_acct|mo_sin_old_rev_tl_op|mo_sin_rcnt_rev_tl_op|mo_sin_rcnt_tl   |mort_acc          |mths_since_recent_bc|
+-------+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------------+------------------+------------------+--------------------+---------------------+-----------------+------------------+--------------------+
|count  |2260668           |2260668           |2260668           |2260668

## Next Step

If the schema looks reasonable, the next practical move is to persist the raw CSV as Parquet for faster repeated analysis:

```python
parquet_path = DATA_DIR / 'loan.parquet'
df.write.mode('overwrite').parquet(str(parquet_path))
```
